# Parte 2: Impacto financiero y JOINS (ejercicios 11 al 16)

In [0]:
%sql
-- Configuración previa
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

## Ejercicio 11. Costo del marketing inútil

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas_canceladas,
    1.50 AS comision_por_reserva,
    ROUND(COUNT(*) * 1.50, 2) AS perdida_total_usd
FROM tb_reservas
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;

**Conclusión:** El indicador cuantifica las comisiones pagadas por reservas canceladas del canal TA/TO, proporcionando evidencia para renegociar los contratos con las agencias de marketing.

## Ejercicio 12. Costo de cancelar tarde

In [0]:
%sql
WITH reservas_fechas AS (
    SELECT
        is_canceled,

        MAKE_DATE(
            CAST(arrival_date_year AS INT),

            CASE arrival_date_month
                WHEN 'January' THEN 1
                WHEN 'February' THEN 2
                WHEN 'March' THEN 3
                WHEN 'April' THEN 4
                WHEN 'May' THEN 5
                WHEN 'June' THEN 6
                WHEN 'July' THEN 7
                WHEN 'August' THEN 8
                WHEN 'September' THEN 9
                WHEN 'October' THEN 10
                WHEN 'November' THEN 11
                WHEN 'December' THEN 12
            END,

            CAST(arrival_date_day_of_month AS INT)
        ) AS arrival_date,

        TO_DATE(
            reservation_status_date,
            'd/MM/yy'
        ) AS cancellation_date

    FROM tb_reservas
),

cancelaciones AS (
    SELECT
        is_canceled,
        arrival_date,
        cancellation_date,

        DATEDIFF(
            arrival_date,
            cancellation_date
        ) AS dias_anticipacion

    FROM reservas_fechas
)

SELECT
    COUNT(*) AS total_cancelaciones_tardias,
    120 AS costo_por_cancelacion_usd,
    COUNT(*) * 120 AS perdida_total_usd

FROM cancelaciones

WHERE is_canceled = 1
  AND dias_anticipacion >= 0
  AND dias_anticipacion < 3;

**Conclusión:** El cálculo permite cuantificar las pérdidas operativas asociadas a cancelaciones de último momento, aportando información para evaluar políticas de cancelación y penalidades.

## Ejercicio 13. LEFT JOIN: reservas y países

In [0]:
%sql
-- TABLA AUXILIAR DE EJEMPLO
CREATE OR REPLACE TEMP VIEW tb_paises AS
SELECT *
FROM VALUES
    ('PRT', 'Portugal'),
    ('ESP', 'España'),
    ('FRA', 'Francia'),
    ('GBR', 'Reino Unido'),
    ('DEU', 'Alemania'),
    ('ITA', 'Italia'),
    ('USA', 'Estados Unidos'),
    ('BRA', 'Brasil')
AS paises(cod_pais, nombre_pais);

In [0]:
%sql
SELECT
    r.country,
    r.is_canceled,
    p.nombre_pais

FROM tb_reservas AS r

LEFT JOIN tb_paises AS p
    ON r.country = p.cod_pais;

**Conclusión:** El cruce permite incorporar información geográfica a las reservas sin excluir registros que todavía no tengan un país identificado en la tabla auxiliar.

## Ejercicio 14. INNER JOIN con agrupación

In [0]:
%sql
SELECT
    p.nombre_pais,
    COUNT(*) AS total_reservas_canceladas

FROM tb_reservas AS r

INNER JOIN tb_paises AS p
    ON r.country = p.cod_pais

WHERE r.is_canceled = 1

GROUP BY p.nombre_pais

ORDER BY total_reservas_canceladas DESC

LIMIT 5;

**Conclusión:** La consulta permite identificar los principales mercados de origen con mayor volumen de cancelaciones, aunque los resultados serán parciales mientras se utilice la tabla ilustrativa de países.

## Ejercicio 15. FULL OUTER JOIN: reservas y vendedores

In [0]:
%sql
-- CREAR TABLA TEMPORAL DE VENDEDORES

CREATE OR REPLACE TEMP VIEW tb_vendedores AS

SELECT *
FROM VALUES
    (1, 'Carlos Ramirez'),
    (2, 'Maria Lopez'),
    (3, 'Juan Perez'),
    (4, 'Ana Torres'),
    (5, 'Luis Fernandez')
AS vendedores(id_vendedor, nombre_vendedor);

In [0]:
%sql
-- CREAR VISTA TEMPORAL DE RESERVAS CON VENDEDORES
CREATE OR REPLACE TEMP VIEW tb_reservas_vendedores AS

WITH reservas AS (
    SELECT
        distribution_channel,
        ROW_NUMBER() OVER (
            ORDER BY distribution_channel, hotel
        ) AS numero_reserva

    FROM cursodataacademy.hotel_db.tb_reservas
)

SELECT
    numero_reserva,
    distribution_channel,

    CASE
        WHEN numero_reserva IN (1, 2) THEN 1
        WHEN numero_reserva IN (3, 4) THEN 2
        WHEN numero_reserva IN (5, 6) THEN 3
        ELSE NULL
    END AS id_vendedor

FROM reservas

WHERE numero_reserva <= 10;

In [0]:
%sql
SELECT
    r.numero_reserva,
    r.distribution_channel,
    v.id_vendedor,
    v.nombre_vendedor,

    CASE
        WHEN v.id_vendedor IS NULL
            THEN 'Reserva sin vendedor'

        WHEN r.numero_reserva IS NULL
            THEN 'Vendedor sin reserva'

        ELSE 'Reserva con vendedor'
    END AS estado_asignacion

FROM tb_reservas_vendedores AS r

FULL OUTER JOIN tb_vendedores AS v
    ON r.id_vendedor = v.id_vendedor

ORDER BY
    estado_asignacion,
    r.numero_reserva;

**Conclusión:** La consulta permite detectar reservas sin un vendedor responsable y vendedores sin reservas asociadas, contribuyendo al control de las asignaciones comerciales.

## Ejercicio 16. RIGHT JOIN: reservas y facturación

In [0]:
%sql
-- CREAR TABLA TEMPORAL DE FACTURAS

CREATE OR REPLACE TEMP VIEW tb_facturas AS

SELECT *
FROM VALUES
    (1, 250.00),
    (2, 480.50),
    (3, 150.00),
    (4, 720.00),
    (5, 310.00),
    (999, 950.00)
AS facturas(id_reserva, monto_facturado);

In [0]:
%sql
-- CREAR VISTA TEMPORAL DE RESERVAS

CREATE OR REPLACE TEMP VIEW tb_reservas_facturacion AS

WITH reservas AS (
    SELECT
        hotel,
        is_canceled,
        adr,

        ROW_NUMBER() OVER (
            ORDER BY hotel, adr
        ) AS id_reserva

    FROM cursodataacademy.hotel_db.tb_reservas
)

SELECT
    id_reserva,
    hotel,
    is_canceled,
    adr

FROM reservas

WHERE id_reserva <= 10;

In [0]:
%sql
SELECT
    f.id_reserva,
    r.hotel,
    f.monto_facturado,

    CASE
        WHEN r.id_reserva IS NULL
            THEN 'Factura sin reserva'
        ELSE 'Factura con reserva'
    END AS estado_facturacion

FROM tb_reservas_facturacion AS r

RIGHT JOIN tb_facturas AS f
    ON r.id_reserva = f.id_reserva

ORDER BY f.id_reserva;

**Conclusión:** El RIGHT JOIN permite identificar facturas sin una reserva asociada, facilitando la detección de inconsistencias en los registros de facturación y contribuyendo al control contable del hotel.